# 03 · PETT crop → polygon extraction  `(geometry, crop, year)`

**Goal.** The cleanest, most reliable table of *(polygon geometry, crop type, year)* — accepting
several rows per polygon when the year differs — using **only the PETT sources joined on real keys**
(no fragile name-matching).

### The chain and the files it needs

```
BD SSET ──CodigoSSET──►  bridge (.dta)  ──COD_PREDIO──►  qgis polygons
(crop, year)             CodigoSSET↔COD_PREDIO           (geometry)
```

BD SSET has no `COD_PREDIO` and the polygon shapefile has no `CodigoSSET`, so a **bridge** table
carrying both keys is mandatory. Two candidate bridges exist (`grafica_tabular_catastro_Piura.dta`
and `grafica_tabular_Piura.dta`); Section 3 shows the latter **supersedes** the former, so we use it.

| file | role | used? |
|---|---|---|
| `BD SSET(...).xlsx` | crop + registration date (→ year) + `Codigo SSET` | ✅ |
| `grafica_tabular_Piura.dta` | bridge `CodigoSSET ↔ COD_PREDIO` | ✅ (best bridge) |
| `qgis_stefany/...FINAL.shp` | polygon geometry, keyed `COD_PREDIO` | ✅ |
| `grafica_tabular_catastro_Piura.dta` | smaller bridge, subset of the above | ❌ redundant (shown in §3) |
| `qgis_stefany/PIURA.dta` | shapefile attribute table, **no `CodigoSSET`** | ❌ useless for bridging (§3) |
| `IV_CENAGRO_Piura.dta`, `Base_...`, crop dict | census route (name-based, uncertain) | ❌ not needed here |

**Important expectation (proven in §6):** SSET is effectively a *single* crop snapshot per parcel
from the ~1998–99 titling wave — almost no polygon carries crops in more than one distinct year, and
the "year" is the *titling* date (batch/stub values), not a growing season. So this yields ~66k
polygons each with one approximate crop-year, not a multi-year time series.

## 1 · Configuration & imports

In [1]:
import warnings
from pathlib import Path
import numpy as np, pandas as pd
import pyreadstat, pyogrio, geopandas as gpd
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 160)

RAW = Path("../data/raw"); OUT = Path("../data/processed"); OUT.mkdir(parents=True, exist_ok=True)
F_SSET   = RAW / "BD SSET(MOQUEGUA-PASCO-PIURA).xlsx"
F_BRIDGE = RAW / "grafica_tabular_Piura.dta"            # chosen bridge (see §3)
F_CAT    = RAW / "grafica_tabular_catastro_Piura.dta"   # compared against, then discarded
F_POLY   = RAW / "qgis_stefany/CATASTRO_CENAGRO_PIURA_WGS84_Z17S_FINAL.shp"
F_PIURA  = RAW / "qgis_stefany/PIURA.dta"               # tested for usefulness (§3)

## 2 · Load BD SSET (Piura) crop declarations

One row per parcel-crop declaration. We keep only rows with a crop label, derive the **year** from
`FECHA EMPADRONAMIENTO`, and canonicalise `Codigo SSET` to a string (it is `int64` here but a string
on the bridge/polygon side).

In [2]:
sset = pd.read_excel(F_SSET, sheet_name="DATOS",
        usecols=["DEPARTAMENTO","CULTIVO","FECHA EMPADRONAMIENTO","AREA","Codigo SSET","NOMBRES"])
sset = sset[sset.DEPARTAMENTO == "PIURA"].copy()
sset = sset.rename(columns={"Codigo SSET":"CodigoSSET","FECHA EMPADRONAMIENTO":"fecha",
                            "CULTIVO":"crop","AREA":"area_m2","NOMBRES":"owner"})
sset["CodigoSSET"] = sset["CodigoSSET"].astype("Int64").astype(str)
sset["fecha"] = pd.to_datetime(sset["fecha"], errors="coerce")
sset["year"]  = sset["fecha"].dt.year
sset_crop = sset[sset.crop.notna()].copy()
print(f"BD SSET Piura rows: {len(sset):,}")
print(f"  with a crop label: {len(sset_crop):,} ({len(sset_crop)/len(sset):.0%}) "
      f"across {sset_crop.CodigoSSET.nunique():,} distinct Codigo SSET")
print(f"  year coverage: {int(sset_crop.year.min())}-{int(sset_crop.year.max())}")
sset_crop.head(3)

BD SSET Piura rows: 348,076
  with a crop label: 177,695 (51%) across 98,348 distinct Codigo SSET
  year coverage: 1996-2019


,DEPARTAMENTO,owner,fecha,area_m2,crop,CodigoSSET,year
7,PIURA,MARTINEZ AQUINO FAUSTINO,1996-01-01,494.0,ARROZ,230134781,1996
8,PIURA,SOSA DE MARTINEZ MARIA LUISA,1996-01-01,494.0,ARROZ,230134781,1996
9,PIURA,VELASCO SUAREZ LEONARDO,1996-01-01,122921.0,CAÑA DE AZUCAR,230049161,1996


## 3 · Choose the bridge — compare `catastro` vs `grafica_tabular_Piura`, and test `PIURA.dta`

The bridge is the **bottleneck**: not every SSET `CodigoSSET` appears in a bridge, so this step
decides how many crop records ever reach a polygon. We compare both candidate bridges on the metric
that matters — *share of SSET crop keys covered* — and check whether `qgis/PIURA.dta` can add
anything.

In [3]:
def load_bridge(path):
    b, _ = pyreadstat.read_dta(str(path), usecols=["COD_PREDIO","CodigoSSET"], encoding="latin1")
    b["COD_PREDIO"] = b.COD_PREDIO.astype(str).str.strip().replace("nan", np.nan)
    b["CodigoSSET"] = b.CodigoSSET.astype(str).str.strip().replace("nan", np.nan)
    return b.dropna().drop_duplicates()

cat    = load_bridge(F_CAT)
bridge = load_bridge(F_BRIDGE)
crop_keys = set(sset_crop.CodigoSSET)

rep = pd.DataFrame({
    "grafica_tabular_catastro": {"pairs": len(cat), "distinct CodigoSSET": cat.CodigoSSET.nunique(),
        "SSET crop keys covered": len(crop_keys & set(cat.CodigoSSET)),
        "coverage %": round(100*len(crop_keys & set(cat.CodigoSSET))/len(crop_keys), 1)},
    "grafica_tabular_Piura": {"pairs": len(bridge), "distinct CodigoSSET": bridge.CodigoSSET.nunique(),
        "SSET crop keys covered": len(crop_keys & set(bridge.CodigoSSET)),
        "coverage %": round(100*len(crop_keys & set(bridge.CodigoSSET))/len(crop_keys), 1)},
}).T
print(rep.to_string())
# is catastro a subset of grafica_tabular_Piura?
union_pairs = pd.concat([cat, bridge]).drop_duplicates()
print(f"\nunion of both bridges = {len(union_pairs):,} pairs; grafica_tabular_Piura alone = {len(bridge):,}")
print("=> catastro adds", len(union_pairs) - len(bridge), "new pairs -> catastro is a SUBSET; keep grafica_tabular_Piura only.")

                             pairs  distinct CodigoSSET  SSET crop keys covered  coverage %
grafica_tabular_catastro  109796.0             109378.0                 58837.0        59.8
grafica_tabular_Piura     158427.0             157857.0                 67639.0        68.8

union of both bridges = 158,427 pairs; grafica_tabular_Piura alone = 158,427
=> catastro adds 0 new pairs -> catastro is a SUBSET; keep grafica_tabular_Piura only.


In [4]:
# Can qgis/PIURA.dta help? It must carry CodigoSSET to bridge, or add COD_PREDIO the shapefile lacks.
_, pmeta = pyreadstat.read_dta(str(F_PIURA), metadataonly=True, encoding="latin1")
piura_cod, _ = pyreadstat.read_dta(str(F_PIURA), usecols=["COD_PREDIO"], encoding="latin1")
piura_set = set(piura_cod.COD_PREDIO.dropna().astype(str).str.strip())
poly_ids = pyogrio.read_dataframe(str(F_POLY), columns=["COD_PREDIO"], read_geometry=False)
poly_set = set(poly_ids.COD_PREDIO.dropna().astype(str).str.strip())
print("qgis/PIURA.dta has 'CodigoSSET'? ->", "CodigoSSET" in pmeta.column_names)
print(f"qgis/PIURA.dta COD_PREDIO overlap with shapefile: {len(piura_set & poly_set):,} / {len(piura_set):,}")
missing = (set(bridge.COD_PREDIO) - poly_set) & piura_set
print(f"bridge predios missing from shapefile that PIURA.dta could recover: {len(missing)}")
print("=> PIURA.dta carries no CodigoSSET and duplicates the shapefile's predios -> useless for bridging.")

qgis/PIURA.dta has 'CodigoSSET'? -> False
qgis/PIURA.dta COD_PREDIO overlap with shapefile: 154,539 / 154,540
bridge predios missing from shapefile that PIURA.dta could recover: 1
=> PIURA.dta carries no CodigoSSET and duplicates the shapefile's predios -> useless for bridging.


**Conclusion.** `grafica_tabular_Piura.dta` covers **~69%** of SSET crop keys vs catastro's
**~60%**, and catastro contributes no pairs beyond it (it is a subset). `qgis/PIURA.dta` has no
`CodigoSSET` and merely repeats the shapefile's `COD_PREDIO`, so it cannot recover anything. We
proceed with **`grafica_tabular_Piura.dta`** as the sole bridge.

## 4 · Load the polygons (geometry)

The shapefile is the geometry source (EPSG:32717). We drop null-code polygons and collapse any
duplicate `COD_PREDIO` so the key is unique, then reproject to EPSG:4326.

In [5]:
poly = pyogrio.read_dataframe(str(F_POLY), columns=["COD_PREDIO","AREA_S_HA"])
poly = poly[poly.COD_PREDIO.notna()].copy()
poly["COD_PREDIO"] = poly.COD_PREDIO.astype(str).str.strip()
if poly.COD_PREDIO.duplicated().any():
    poly = poly.dissolve(by="COD_PREDIO", aggfunc="first").reset_index()
poly = poly.to_crs(4326)
print(f"polygons: {len(poly):,} unique COD_PREDIO · CRS {poly.crs}")

polygons: 179,608 unique COD_PREDIO · CRS EPSG:4326


## 5 · Build the `(COD_PREDIO, crop, year)` table

Join SSET crops → bridge → require the predio to have a polygon. A `CodigoSSET` maps to essentially
one `COD_PREDIO` (a handful map to >1; we keep all). We report both the raw record count and the
**distinct** `(COD_PREDIO, crop, year)` set (collapsing redundant re-declarations).

In [6]:
linked = (sset_crop.merge(bridge, on="CodigoSSET", how="inner"))
linked = linked[linked.COD_PREDIO.isin(set(poly.COD_PREDIO))].copy()
print(f"crop records reaching a polygon: {len(linked):,} · distinct polygons: {linked.COD_PREDIO.nunique():,}")

crop_year = (linked[["COD_PREDIO","CodigoSSET","crop","year","area_m2","fecha","owner"]]
             .sort_values(["COD_PREDIO","year"]))
# distinct (polygon, crop, year) — the clean deliverable grain
distinct_cy = crop_year.drop_duplicates(["COD_PREDIO","crop","year"])
print(f"distinct (polygon, crop, year) rows: {len(distinct_cy):,}")
crop_year.head(5)

crop records reaching a polygon: 116,599 · distinct polygons: 66,363
distinct (polygon, crop, year) rows: 68,717


,COD_PREDIO,CodigoSSET,crop,year,area_m2,fecha,owner
93547,7_4909455_50548,230090683,ALGODON,2001,60952.0,2001-01-01,LUNA MASIAS DE CALDERON LUZ CONSUELO
93548,7_4909455_50548,230090683,ALGODON,2001,60952.0,2001-01-01,LUNA MASIAS ALEJANDRO
93549,7_4909455_50548,230090683,ALGODON,2001,60952.0,2001-01-01,LUNA MASIAS ELEOVINA
93550,7_4909455_50548,230090683,ALGODON,2001,60952.0,2001-01-01,LUNA MASIAS DE PEÑARRIETA ISABEL
93551,7_4909455_50548,230090683,ALGODON,2001,60952.0,2001-01-01,LUNA MASIAS DE MIMBELA CARMEN MELANIA


## 6 · Reality check: how many polygons actually have *multiple* crop-years?

You accepted several rows per polygon when the year differs. This section measures whether that
actually happens, and shows the year distribution so the reliability of the "year" is explicit.

In [7]:
ny = distinct_cy.groupby("COD_PREDIO").year.nunique()
nc = distinct_cy.groupby("COD_PREDIO").apply(lambda d: d[["crop","year"]].drop_duplicates().shape[0])
print("polygons by number of DISTINCT years present:")
print(ny.value_counts().sort_index().to_string())
print(f"\npolygons with >1 distinct crop-year: {(ny>1).sum():,} of {ny.size:,} "
      f"({(ny>1).mean():.2%})")
print("\ncrop-year records by year (titling wave):")
print(distinct_cy.year.value_counts().sort_index().to_string())

polygons by number of DISTINCT years present:
year
1    66362
2        1

polygons with >1 distinct crop-year: 1 of 66,363 (0.00%)

crop-year records by year (titling wave):
year
1996        7
1998    25037
1999    23364
2000     5973
2001     2727
2002     1168
2003     3600
2004      340
2005     2908
2006     1861
2007     1103
2008      284
2009      284
2010       48
2011        2
2015        8
2019        3


As expected the crop years collapse onto the ~1998–99 titling campaign; a negligible number
of polygons carry crops in more than one year. Treat this as **one crop label per polygon**, with
the year being the registration/titling date rather than a verified growing season.

## 7 · Save outputs (lean)

* **`pett_polygons.parquet`** — one geometry per `COD_PREDIO` (join target).
* **`pett_crop_year.parquet`** — the `(COD_PREDIO, crop, year, …)` records; join to the polygons on
  `COD_PREDIO`. Also a `distinct` view collapsing redundant re-declarations.

In [8]:
p_poly = OUT / "pett_polygons.parquet"
p_cy   = OUT / "pett_crop_year.parquet"
p_cyd  = OUT / "pett_crop_year_distinct.parquet"
used_poly = poly[poly.COD_PREDIO.isin(set(crop_year.COD_PREDIO))][["COD_PREDIO","geometry","AREA_S_HA"]]
used_poly.to_parquet(p_poly)
crop_year.to_parquet(p_cy, index=False)
distinct_cy.to_parquet(p_cyd, index=False)
for p, df in [(p_poly, used_poly), (p_cy, crop_year), (p_cyd, distinct_cy)]:
    print(f"wrote {p.name:32s} {p.stat().st_size/1e6:5.1f} MB · {len(df):,} rows")

# quick demo: a ready-to-use GeoDataFrame at crop-record grain
demo = used_poly.merge(distinct_cy, on="COD_PREDIO").head(3)
print("\nexample joined rows:")
print(demo[["COD_PREDIO","crop","year","AREA_S_HA"]].to_string(index=False))

wrote pett_polygons.parquet             20.6 MB · 66,363 rows
wrote pett_crop_year.parquet             2.4 MB · 116,599 rows
wrote pett_crop_year_distinct.parquet    1.8 MB · 68,717 rows

example joined rows:
      COD_PREDIO                crop  year  AREA_S_HA
 7_4909455_50548             ALGODON  2001   6.063574
 7_4909455_50624 TERRENO EN DESCANSO  2001   2.372554
7_4959460_066715               ARROZ  2005   4.974098


### Summary

* **Best 3-file chain:** `BD SSET` + **`grafica_tabular_Piura.dta`** (bridge) + `qgis` polygons →
  **~116,600 crop records over ~66,400 polygons**. Using `grafica_tabular_Piura` instead of
  `catastro` recovers **~+12,800 records / +8,400 polygons**; `catastro` and `qgis/PIURA.dta` are
  redundant.
* **Reliable link** (real keys throughout) — unlike the census route, no name-matching.
* **One crop-year per polygon in practice**, from the ~1998–99 titling wave; the "year" is the
  registration date (batch/stub), so use it as an approximate label, not a precise season. A genuine
  second time point per polygon only exists in the CENAGRO census (2012) via the uncertain
  name-based link (see notebook 02).